# Data Merging Sandbox

This is where I will work to merge datasets for different sites (Gothic and Kettle Ponds) and external gridded datasets at hourly to daily resolutions.

In [5]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
if project_root not in sys.path:
    sys.path.append(project_root)
from utils import process_sail_data

DATA_PATH = '/storage/dlhogan/precipitation-rodeo/data/processed/'

# Sand Castle 1 - Gothic Precipitation

In [55]:
# Load datasets from Gothic
billy_barr_ds = xr.open_dataset(f'{DATA_PATH}billy_barr/billy_barr_20211001_20230930_30min.nc')['precip'].to_dataset().sortby('time')
sail_ld_ds = xr.open_dataset(f'{DATA_PATH}SAIL/laser_disdrometer_gothic_processed_30min.nc')[['precip_accum_uncorrected','precip_accum_holyroyd',
                                                                                             'precip_accum_brandes','precip_accum_heymsfield',]].sortby('time')
sail_pluvio_ds = xr.open_dataset(f'{DATA_PATH}SAIL/pluvio_30min.nc')['accum_nrt'].to_dataset().sortby('time')
sail_met_ds = xr.open_dataset(f'{DATA_PATH}SAIL/met_30min.nc').sortby('time')
sail_squire_ds = xr.open_dataset(f'{DATA_PATH}SAIL/squire_30min.nc').sel(site='gothic').sortby('time').squeeze()

# drop all vars in sail_met_ds not in process_sail_data.SAIL_PRECIPITATION_VARS['cumulative'] 
met_prcp_vars = [var for var in process_sail_data.SAIL_PRECIPITATION_VARS['cumulative'] if var in sail_met_ds.data_vars]
squire_prcp_vars = [var for var in process_sail_data.SAIL_PRECIPITATION_VARS['cumulative'] if var in sail_squire_ds.data_vars]
sail_squire_ds = sail_squire_ds[squire_prcp_vars].drop_vars(['lat','lon','x','y','site'])
sail_met_ds = sail_met_ds[met_prcp_vars]

In [56]:
print('Merging datasets...')
try:
    gothic_combined_ds = xr.merge([billy_barr_ds, sail_ld_ds, sail_pluvio_ds, sail_met_ds, sail_squire_ds], compat='override')
    print('Datasets merged successfully.')
except Exception as e:
    print(f'Error merging datasets: {e}')

Merging datasets...
Datasets merged successfully.


In [ ]:
# rename the variables
variable_renames = {
    'precip': 'billy_barr_precip',
    'precip_accum_uncorrected': 'sail_ld_uncorrected',
    'precip_accum_holyroyd':'sail_ld_holyroyd',
    'precip_accum_brandes':'sail_ld_brandes',
    'precip_accum_heymsfield':'sail_ld_heymsfield',
    'accum_nrt':'sail_pluvio',
    'pwd_cumul_rain':'sail_pwd_rain',
    'pwd_cumul_snow':'sail_pwd_snow',
    'tbrg_precip_total':'sail_tbg',
    'tbrg_precip_total_corr':'sail_tbg_corr',
    'org_precip_accum':'sail_org',
    'rain_rate_A_total':'sail_squire_rain',
    'snow_rate_m2009_1_total':'sail_squire_snow_m2009_1',
    'snow_rate_m2009_2_total':'sail_squire_snow_m2009_2',
    'snow_rate_ws88diw_total':'sail_squire_snow_ws88diw',
    'snow_rate_ws2012_total':'sail_squire_snow_ws2012'
}

gothic_combined_ds = gothic_combined_ds.rename(variable_renames)

# Sand Castle 2 - Kettle Ponds Precipitation

In [ ]:
# Load datasets from Kettle p\Ponds
splash_lpdf_ds = xr.open_dataset(f'{DATA_PATH}SPLASH/lpdf_gauge_30min.nc')
splash_ld_ds = xr.open_dataset(f'{DATA_PATH}SPLASH/SPLASH_kp_laser_disdrometer_30min.nc')[['Amount']]
sos_ds = xr.open_dataset(f'{DATA_PATH}SOS/sos_ds_30min.nc')[['SWE_p1_c_max_accum','SWE_p2_c_max_accum',
                                                             'SWE_p3_c_max_accum','SWE_p4_c_max_accum',]]

In [66]:
# merge kettle ponds datasets
print('Merging Kettle Ponds datasets...')
try:
    kettle_ponds_combined_ds = xr.merge([splash_lpdf_ds, splash_ld_ds, sos_ds], compat='override')
    print('Kettle Ponds datasets merged successfully.')
except Exception as e:
    print(f'Error merging Kettle Ponds datasets: {e}')

Merging Kettle Ponds datasets...
Kettle Ponds datasets merged successfully.


In [ ]:
variable_renames_kp = {
    'prcp':'splash_lpdf',
    'Amount': 'splash_ld_uncorrected',
    'SWE_p1_c_max_accum': "sos_swe_p1",
    'SWE_p2_c_max_accum': "sos_swe_p2",
    'SWE_p3_c_max_accum': "sos_swe_p3",
    'SWE_p4_c_max_accum': "sos_swe_p4"
}

kettle_ponds_combined_ds = kettle_ponds_combined_ds.rename(variable_renames_kp)